# AKE Lab 1 · MLP für MNIST — Starter-Notebook

**Angewandte KI & KI-Engineering · Block A – Vertiefung Deep Learning · Woche 1**
Zugehöriges Skript: *Vom Perzeptron zum Deep Network*, Abschnitt 9 · Aufgabenblatt: `AKE_Lab01_Aufgabenblatt.pdf`

Arbeiten Sie die Zellen der Reihe nach ab. Stellen mit `TODO` bzw. `...` sind von Ihnen zu ergänzen. Nach jedem Experiment folgt eine Markdown-Zelle **Interpretation** — dort erwarten wir 3–6 Sätze, die Ihr Ergebnis mit dem Skript verknüpfen (Gleichungs- oder Tabellennummer angeben).

**Rechenbudget (Basis-Lab, Referenzhardware 4 Kerne / 8 GB, keine GPU):** reine Rechenzeit ≈ 8–12 Min. Alle
Experimente schreiben ihre Trainingslogs als JSON nach `results/`. Bricht ein eigener Lauf ab, können Sie die
mitgelieferten Referenzlogs aus `results_referenz/` laden und die Analyse trotzdem bearbeiten.


## 0 · Setup und Daten

Importe, Pfade, MNIST-Splits (54 000 / 6 000 / 10 000) wie in Skript Listing 3. Der Testsplit wird **ausschließlich** für die finale Bewertung je Aufgabe benutzt – nie zur Modellauswahl (Abschn. 6.1).

In [ ]:
import os, time, json, math, copy
from pathlib import Path

import numpy as np
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)

# MNIST liegt auf dem Hochschul-Share unter <Share>/data/MNIST. Ordner nach ./data kopieren
# (oder AKE_DATA auf den Share-Pfad setzen). Fehlt er, laedt torchvision die ~12 MB herunter.
DATA_ROOT = os.environ.get("AKE_DATA", "data")
RESULTS   = Path("results"); RESULTS.mkdir(exist_ok=True)   # Trainingslogs je Experiment (JSON)

print("torch", torch.__version__, "| Threads:", torch.get_num_threads(), "| CUDA:", torch.cuda.is_available())

In [ ]:
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.1307,), (0.3081,))])   # Mittelwert/Std von MNIST

full = datasets.MNIST(DATA_ROOT, train=True,  download=True, transform=tf)
test = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=tf)
train, val = random_split(full, [54_000, 6_000], generator=torch.Generator().manual_seed(0))

def make_loaders(train_ds=train, batch_size=64):
    """Train-Loader wird gemischt, Val/Test nicht. Val/Test sind immer die vollen Splits."""
    return {"train": DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            "val":   DataLoader(val,      batch_size=256),
            "test":  DataLoader(test,     batch_size=256)}

dl = make_loaders()
print(f"Train {len(train)} | Val {len(val)} | Test {len(test)} | Batches/Epoche: {len(dl['train'])}")

fig, ax = plt.subplots(1, 8, figsize=(8, 1.4))
for i in range(8):
    x, y = full[i]
    ax[i].imshow(x[0], cmap="gray_r"); ax[i].set_title(y, fontsize=9); ax[i].axis("off")
plt.show()

### Modell, Trainings- und Evaluationsschleife

`MLP` verallgemeinert Listing 1 (variable Tiefe, Breite, Aktivierung, optional Dropout). `run_epoch`/`fit` sind die Referenzschleife aus Listing 3, ergänzt um Logging, Early Stopping und JSON-Export. Diese Schleife bleibt im Rest des Semesters praktisch unverändert (Abschn. 9.3).

In [ ]:
class MLP(nn.Module):
    """MLP 784 -> hidden[0] -> ... -> n_classes. Ausgabe sind Logits (Softmax steckt im Loss).
    act: Aktivierungsklasse (nn.ReLU, nn.Sigmoid, ...); dropout: p nach jeder Aktivierung."""
    def __init__(self, d_in=784, hidden=(128, 128), n_classes=10, act=nn.ReLU, dropout=0.0, init="default"):
        super().__init__()
        layers, d = [nn.Flatten()], d_in
        for h in hidden:
            # TODO: nn.Linear(d, h) und act() anhaengen; bei dropout > 0 zusaetzlich nn.Dropout(dropout)
            #       (Reihenfolge laut Skript Abschn. 6.4: Linear -> Aktivierung -> Dropout)
            ...
            d = h
        # TODO: Ausgabeschicht nn.Linear(d, n_classes) anhaengen -- OHNE Softmax (Skript Abschn. 3.4)
        ...
        self.net = nn.Sequential(*layers)
        self.reset_parameters(init)

    def reset_parameters(self, init):
        """init: "default" (PyTorch: kaiming_uniform mit a=sqrt(5), d.h. Var = 1/(3 n_in)),
        "glorot" (Var = 2/(n_in+n_out), Gl. 24) oder "he" (Var = 2/n_in, Gl. 25). Biases = 0."""
        if init == "default":
            return
        for m in self.net:
            if isinstance(m, nn.Linear):
                {"glorot": nn.init.xavier_normal_,
                 "he": lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu")}[init](m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

model = MLP()
n_params = sum(p.numel() for p in model.parameters())
print(model.net)
print("Parameter:", n_params)
assert n_params == 118_282, "Parameterzahl weicht von Skript Bsp. 3.2 ab -- Architektur pruefen"

In [ ]:
lossf = nn.CrossEntropyLoss()   # erwartet Logits, kein Softmax davor!

def run_epoch(model, loader, opt=None):
    """Eine Epoche. opt=None => reine Evaluation (kein Gradient, eval()-Modus)."""
    train_mode = opt is not None
    model.train(train_mode)
    tot_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train_mode):
        for x, y in loader:
            logits = model(x)
            loss = lossf(logits, y)
            if train_mode:
                opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * len(y)
            correct  += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return tot_loss / n, correct / n

def fit(model, opt, loaders, epochs, name=None, patience=None, verbose=True):
    """Trainiert `epochs` Epochen, loggt Train-/Val-Verlust und -Genauigkeit.
    patience=k: Early Stopping — Abbruch nach k Epochen ohne Verbesserung des Val-Verlusts,
    beste Parameter werden wiederhergestellt. Rueckgabe: history-Dict (auch als JSON in results/)."""
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_time": []}
    best_loss, best_state, bad = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr = run_epoch(model, loaders["train"], opt)
        va = run_epoch(model, loaders["val"])
        for k, v in zip(["train_loss", "train_acc", "val_loss", "val_acc"], (*tr, *va)):
            hist[k].append(v)
        hist["epoch_time"].append(time.time() - t0)
        if verbose:
            print(f"  Epoche {ep:3d}: train loss {tr[0]:.4f} acc {tr[1]:.2%} | "
                  f"val loss {va[0]:.4f} acc {va[1]:.2%} | {hist['epoch_time'][-1]:.1f}s")
        if not math.isfinite(va[0]):
            print("  -> Verlust nicht endlich, Abbruch."); break
        if va[0] < best_loss:
            best_loss, bad = va[0], 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if patience is not None and bad >= patience:
                print(f"  -> Early Stopping nach Epoche {ep} (bestes Val-Loss {best_loss:.4f} "
                      f"in Epoche {ep - bad}).")
                break
    if patience is not None and best_state is not None:
        model.load_state_dict(best_state)          # Parameterstand am Val-Minimum behalten
    hist["best_epoch"] = int(np.argmin(hist["val_loss"])) + 1
    if name:
        (RESULTS / f"{name}.json").write_text(json.dumps(hist, indent=1))
    return hist

def plot_history(hists, title="", metric="loss", ax=None, logy=False):
    """hists: {label: history}. Zeichnet Train (durchgezogen) und Val (gestrichelt)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 3.6))
    for i, (label, h) in enumerate(hists.items()):
        c = f"C{i}"
        ax.plot(range(1, len(h[f"train_{metric}"]) + 1), h[f"train_{metric}"], c, label=f"{label} (train)")
        ax.plot(range(1, len(h[f"val_{metric}"]) + 1),   h[f"val_{metric}"],   c + "--", label=f"{label} (val)")
    ax.set_xlabel("Epoche"); ax.set_ylabel(metric); ax.set_title(title); ax.grid(alpha=.3)
    if logy: ax.set_yscale("log")
    ax.legend(fontsize=7)
    return ax

## Aufgabe 1 · Baseline

MLP 784–128–128–10 (ReLU), Adam $\alpha=10^{-3}$, Batch 64, 5 Epochen auf den vollen 54 000 Trainingsbildern. Erwartung: ≈ 97–98 % Testgenauigkeit. Vorher der erste Schritt des Debug-Protokolls (Abschn. 8): Verlust beim ersten Batch ≈ ln 10.

In [ ]:
torch.manual_seed(0)
model = MLP(hidden=(128, 128))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Debug-Protokoll Schritt 1 (Skript Abschn. 8): Verlust vor dem Training ~ ln(10)?
x0, y0 = next(iter(dl["train"]))
print(f"Verlust vor Training: {lossf(model(x0), y0).item():.3f}  (erwartet ~ ln 10 = {math.log(10):.3f})")

# TODO: 5 Epochen trainieren (fit), anschliessend auf dl["test"] evaluieren (run_epoch ohne opt)
h_base = ...
test_loss, test_acc = ...
print(f"Test: loss {test_loss:.4f} | acc {test_acc:.2%}")

# TODO: Trainings- und Validierungsverlust je Epoche plotten (plot_history)


### Interpretation Aufgabe 1

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Aufgabe 2 · Architektur und Vanishing Gradients

10-schichtiges Netz (10 × 100 Einheiten), einmal mit Sigmoid, einmal mit ReLU, SGD $\epsilon = 0{,}1$. Nach dem ersten Batch die Gradientennorm der ersten Schicht messen (`model.net[1].weight.grad.norm()`) und mit Gl. (29) erklären. Anschließend prüfen, ob die Netze überhaupt lernen.

In [ ]:
def grad_norms_first_batch(model, lr=0.1):
    """Vorwaerts- und Rueckwaertsdurchlauf auf dem ersten Batch (ohne Parameter-Update).
    Rueckgabe: (Loss, ||dL/dz_l|| je Aktivierung -- der Adjungierte aus Gl. 29, wie Abb. 17 --,
    ||dL/dW_l|| je Linear-Schicht)."""
    acts = []
    def keep(module, inp, out):          # Forward-Hook: Aktivierung merken und ihren Gradienten behalten
        out.retain_grad(); acts.append(out)
    hooks = [m.register_forward_hook(keep) for m in model.net if not isinstance(m, (nn.Linear, nn.Flatten))]
    x, y = next(iter(dl["train"]))
    model.zero_grad()
    # TODO: Vorwaertsdurchlauf, Loss, backward()
    ...
    for h in hooks: h.remove()
    z_norms = [a.grad.norm().item() for a in acts]                      # Adjungierte je Aktivierung
    # TODO: Liste der Gradientennormen ||dL/dW_l|| aller nn.Linear-Schichten in model.net
    #       (Tipp: m.weight.grad.norm() fuer m in model.net, falls isinstance(m, nn.Linear))
    w_norms = ...
    return loss.item(), z_norms, w_norms

deep = {}
for act in (nn.Sigmoid, nn.ReLU):
    torch.manual_seed(0)
    m = MLP(hidden=(100,) * 10, act=act, init="glorot")        # 10 verdeckte Schichten a 100, Glorot wie Abb. 17
    loss0, zn, wn = grad_norms_first_batch(m)
    deep[act.__name__] = {"model": m, "z_norms": zn, "w_norms": wn, "loss0": loss0}
    print(f"{act.__name__:8s}: model.net[1].weight.grad.norm() = {wn[0]:.2e} | "
          f"||dL/dz_1|| = {zn[0]:.2e} | ||dL/dz_10|| = {zn[-1]:.2e}")

# TODO: ||dL/dz_l|| ueber der Schichtnummer plotten (log-Skala!), beide Aktivierungen in einem Plot


In [ ]:
# Lernt das tiefe Netz ueberhaupt? 3 Epochen auf einem 10 000er-Subset, SGD lr=0.1
sub10k = Subset(train, range(10_000))
dl10k = make_loaders(sub10k)
h_deep = {}
for name, d in deep.items():
    print(name)
    # TODO: SGD-Optimierer (lr=0.1) auf d["model"], 3 Epochen mit fit(..., loaders=dl10k)
    h_deep[name] = ...

# TODO: beide Verlaeufe plotten; Referenzlinie ln(10) einzeichnen (plt.axhline)


### Interpretation Aufgabe 2

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Aufgabe 3 · Optimierer × Lernrate

SGD, SGD+Momentum (0,9) und Adam mit Lernraten $\{10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}\}$, je 2 Epochen. **Rechenbudget:** Der Sweep läuft auf dem 10 000er-Subset `dl10k` aus Aufgabe 2 (12 Läufe × 2 Epochen ≈ 2–3 Min. auf 4 Kernen); validiert wird auf dem vollen Validierungssplit. Welche Kombination erreicht den geringsten Validierungsverlust, welche divergiert?

In [ ]:
OPTIMIZERS = {
    "SGD":          lambda p, lr: torch.optim.SGD(p, lr=lr),
    "SGD+Momentum": lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9),
    "Adam":         lambda p, lr: torch.optim.Adam(p, lr=lr),
}
LRS = [1e-4, 1e-3, 1e-2, 1e-1]

sweep = {}
for oname, make_opt in OPTIMIZERS.items():
    for lr in LRS:
        torch.manual_seed(0)              # gleicher Startpunkt fuer alle Laeufe!
        m = MLP(hidden=(128, 128))
        # TODO: 2 Epochen auf dl10k trainieren (verbose=False), history in sweep[(oname, lr)] ablegen
        ...
        # TODO: val loss nach 2 Epochen ausgeben; nicht endliche Werte (NaN/inf) als "divergiert" markieren

# TODO: Kombination mit dem geringsten Validierungsverlust bestimmen


In [ ]:
# TODO: Validierungsverlust ueber den Epochen fuer alle 12 Kombinationen plotten
#       (ein Subplot je Optimierer, eine Linie je Lernrate, log-Skala) und als 3x4-Tabelle ausgeben


### Interpretation Aufgabe 3

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Aufgabe 4 · Regularisierungs-Ablation

Training auf nur 1 000 Beispielen (künstliche Datenknappheit), 50 Epochen, Adam $10^{-3}$. Vergleich: keine Regularisierung / Weight Decay $10^{-3}$ / Dropout 0,5 / Early Stopping (patience 5). Begründen Sie mit Tab. 4 des Skripts, welche Kombination Sie empfehlen. Die Testgenauigkeit wird je Variante genau einmal am Ende ermittelt.

In [ ]:
sub1k = Subset(train, range(1_000))                # kuenstliche Datenknappheit
dl1k = make_loaders(sub1k)                          # 16 Batches/Epoche, Val/Test bleiben voll

VARIANTS = {
    "keine Regularisierung":   dict(dropout=0.0, weight_decay=0.0,  patience=None),
    "Weight Decay 1e-3":       dict(dropout=0.0, weight_decay=1e-3, patience=None),
    "Dropout 0,5":             dict(dropout=0.5, weight_decay=0.0,  patience=None),
    "Early Stopping (pat. 5)": dict(dropout=0.0, weight_decay=0.0,  patience=5),
}

reg = {}
for name, cfg in VARIANTS.items():
    torch.manual_seed(0)
    # TODO: MLP(hidden=(128,128), dropout=cfg["dropout"]) anlegen,
    #       Adam(lr=1e-3, weight_decay=cfg["weight_decay"]),
    #       50 Epochen auf dl1k mit patience=cfg["patience"] trainieren (verbose=False),
    #       anschliessend Test-Genauigkeit in h["test_acc"] ablegen und reg[name] = h
    ...
    print(f"{name}: Epochen {len(h['val_loss'])} | bestes val loss {min(h['val_loss']):.4f} | Test acc {h['test_acc']:.2%}")


In [ ]:
# TODO: Trainings- (hell) und Validierungsverlust (gestrichelt) aller Varianten in einem Plot,
#       Validierungsgenauigkeit in einem zweiten; Ergebnistabelle mit Epochenzahl, bestem val loss, Test-Genauigkeit


### Interpretation Aufgabe 4

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Aufgabe 5 (optional) · Backpropagation in NumPy

Implementieren Sie den Rückwärtsdurchlauf für das einschichtige MLP aus Bsp. 4.1 des Skripts (ReLU + Softmax/Cross-Entropy) und verifizieren Sie die Gradienten per zentraler finiter Differenzen auf einem winzigen Netz in `float64`.

In [ ]:
def relu(z):  return np.maximum(z, 0)

def forward_np(params, X, y):
    """Einschichtiges MLP (Skript Bsp. 4.1): z = W x + b1, h = ReLU(z), a = V h + b2, L = mittlere CE.
    X: (N, D), y: (N,) int. Rueckgabe: Loss und Cache fuer den Rueckwaertsdurchlauf."""
    W, b1, V, b2 = params["W"], params["b1"], params["V"], params["b2"]
    Z = X @ W.T + b1                     # (N, K)
    H = relu(Z)                          # (N, K)
    A = H @ V.T + b2                     # (N, C) Logits
    A_shift = A - A.max(1, keepdims=True)                      # numerisch stabile Softmax
    logp = A_shift - np.log(np.exp(A_shift).sum(1, keepdims=True))
    loss = -logp[np.arange(len(y)), y].mean()
    return loss, dict(X=X, Z=Z, H=H, P=np.exp(logp), y=y)

def backward_np(params, cache):
    """Adjungierte von hinten nach vorne (Skript Gl. 13-15, Bsp. 4.1). Beachten Sie die Mittelung ueber N."""
    X, Z, H, P, y = cache["X"], cache["Z"], cache["H"], cache["P"], cache["y"]
    N = len(y)
    # TODO: U2 = dL/da = (p_hat - y_onehot) / N                        (Gl. 14)
    # TODO: grads["V"] = U2^T H, grads["b2"] = Summe von U2 ueber N    (Gl. 15)
    # TODO: U1 = (U2 V) * 1[Z > 0]                                     (ReLU-VJP)
    # TODO: grads["W"] = U1^T X, grads["b1"] = Summe von U1 ueber N
    grads = {}
    ...
    return grads

def finite_diff(params, X, y, eps=1e-5):
    """Zentrale Differenzen fuer jeden Parameter -- nur fuer winzige Netze bezahlbar."""
    num = {}
    for k, p in params.items():
        g = np.zeros_like(p)
        for idx in np.ndindex(p.shape):
            old = p[idx]
            p[idx] = old + eps; lp, _ = forward_np(params, X, y)
            p[idx] = old - eps; lm, _ = forward_np(params, X, y)
            p[idx] = old
            g[idx] = (lp - lm) / (2 * eps)
        num[k] = g
    return num

rng = np.random.default_rng(0)
D, K, C, N = 5, 4, 3, 7
params = {"W": rng.normal(0, 1/np.sqrt(D), (K, D)), "b1": rng.normal(0, .1, K),
          "V": rng.normal(0, 1/np.sqrt(K), (C, K)), "b2": rng.normal(0, .1, C)}
X = rng.normal(size=(N, D)); y = rng.integers(0, C, N)

loss, cache = forward_np(params, X, y)
ana = backward_np(params, cache)
num = finite_diff(params, X, y)
for k in params:
    print(f"  {k:3s}: max |analytisch - numerisch| = {np.abs(ana[k]-num[k]).max():.2e}   (Ziel: < 1e-7)")

### Interpretation Aufgabe 5

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Abgabe-Checkliste

- [ ] Notebook läuft mit *Kernel → Restart & Run All* fehlerfrei durch (oder: abgebrochene Läufe sind markiert und durch Referenzlogs ersetzt)
- [ ] Zu jeder Aufgabe: Kurven **und** Interpretation mit Skript-Bezug
- [ ] `results/` mit den JSON-Logs liegt bei (Seed, Konfiguration, Kurven – Abschn. 8: *Loggen Sie jeden Lauf*)
- [ ] Dateiname `lab01_<nachname>.ipynb`, Abgabe laut Aufgabenblatt